# Parameter Golf — Colab / Kaggle Launcher

**What this notebook does:** Clones the public parameter-golf repo, downloads data, and runs `train_gpt.py` via torchrun. GPU config is auto-selected by compute capability.

**Works on:** Google Colab and Kaggle Notebooks. Auto-detects environment and GPU.

> **Kaggle:** Enable *Settings → Internet* before running Cell 4.

## GPU tiers

| GPU | Compute cap | Script | Model | Expected BPB (10 min) |
|-----|-------------|--------|-------|-----------------------|
| T4 (free) | SM75 | root `train_gpt.py` | 6L 384d MHA | ~1.5–1.7 |
| P100 (Kaggle free) | SM60 | root `train_gpt.py` | 6L 384d MHA | ~1.4–1.6 |
| A100 / L4 / G4 | SM80–89 | root `train_gpt.py` | 9L 512d GQA | ~1.2–1.35 |
| H100 / Blackwell | SM90–120 | root `train_gpt.py` | 11L 512d GQA | ~1.15–1.30 |

## Known limitations on Colab/Kaggle
- Records scripts (GPTQ int6, sliding-window eval) require pushing local files to GitHub first
- torch.compile disabled on T4/P100 (no bfloat16) and Blackwell (Triton not yet tuned for SM120)
- Flash SDPA patched to math mode on pre-Ampere GPUs after clone


In [ ]:
# Cell 2: Check GPU
!nvidia-smi | head -12
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f"\nGPU: {gpu}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cell 3: Install dependencies
import torch

cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(f"GPU: {gpu_name}  compute_cap: {cc[0]}.{cc[1]}")

!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

# Try Flash Attention 3 on H100 (SM90 only — FA3 wheels have no SM120 kernels)
# FA3 enables the records script (int6+lzma, 11L fits in 16MB, GPTQ, QAT, SWA)
fa3_installed = False
if cc[0] == 9:
    print("H100 detected — attempting FA3 install...")
    try:
        import subprocess, sys
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', 'flash_attn_3',
             '--find-links',
             'https://windreamer.github.io/flash-attention3-wheels/cu128_torch291'],
            capture_output=True, text=True, timeout=300)
        from flash_attn_interface import flash_attn_func
        fa3_installed = True
        print("FA3 installed ✓ — will use records script (int6+lzma, 11L, GPTQ)")
    except Exception as e:
        print(f"FA3 install failed: {e}")
        print("Falling back to root train_gpt.py (int8+zlib, smaller model)")
else:
    print("Non-H100 GPU — FA3 not applicable, using root train_gpt.py")


In [ ]:
# Cell 4: Detect environment, clone/update repo, patch for GPU
import os, subprocess, torch

# Auto-detect Colab vs Kaggle
if os.path.isdir('/content'):
    BASE = '/content'
    print('Environment: Google Colab')
elif os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
    print('Environment: Kaggle')
    print('NOTE: Make sure Internet is ON in Settings → Internet')
else:
    BASE = os.path.expanduser('~')
    print(f'Environment: unknown, using {BASE}')

REPO = f'{BASE}/parameter-golf'
FORK = 'https://github.com/hwangmic-mueon/parameter-golf.git'

# Clone if missing; pull latest if already present
if os.path.isdir(REPO):
    current_remote = subprocess.check_output(
        ['git', '-C', REPO, 'remote', 'get-url', 'origin'], text=True).strip()
    if 'hwangmic-mueon' not in current_remote:
        import shutil, os as _os
        _os.chdir('/')
        shutil.rmtree(REPO)
        print(f"Stale clone ({current_remote}) — recloning")

if not os.path.isdir(REPO):
    !git clone {FORK} {REPO}
else:
    !git -C {REPO} pull origin main
!git -C {REPO} log --oneline -3
%cd {REPO}

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
ampere_or_newer = cc[0] >= 8
print(f'GPU: {gpu_name}  compute_cap: {cc[0]}.{cc[1]}  flash_sdp_ok: {ampere_or_newer}')

script = f'{REPO}/train_gpt.py'
txt = open(script).read()
patched = []

# ── Patch 1: flash→math SDPA for T4/P100 (pre-Ampere, compute cap < 8) ──────
if not ampere_or_newer:
    txt = txt.replace('enable_flash_sdp(True)',  'enable_flash_sdp(False)')
    txt = txt.replace('enable_math_sdp(False)',  'enable_math_sdp(True)')
    patched.append('flash SDPA → math SDPA')

# ── Patch 2: Skip DDP on single-GPU (static_graph=True + compile crashes) ────
OLD_DDP  = ', static_graph=True) if distributed else compiled_model'
NEW_DDP  = ', static_graph=True) if distributed and world_size > 1 else compiled_model'
OLD_DDP2 = 'broadcast_buffers=False) if distributed else compiled_model'
NEW_DDP2 = 'broadcast_buffers=False) if distributed and world_size > 1 else compiled_model'
if OLD_DDP in txt:
    txt = txt.replace(OLD_DDP, NEW_DDP); patched.append('DDP skipped on single-GPU')
elif OLD_DDP2 in txt:
    txt = txt.replace(OLD_DDP2, NEW_DDP2); patched.append('DDP skipped on single-GPU')

if patched:
    open(script, 'w').write(txt)
    for p in patched: print(f'Patched: {p}')
else:
    print('No patches needed')


In [ ]:
# Cell 5: Download data (~800MB, 3-5 min)
# --train-shards 1 is fine for smoke tests. Use --train-shards 80 for real runs.
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1


In [ ]:
# Cell 5b: Model size estimator — run before Cell 6 to check budget
# Calibrated against two measured runs:
#   8L 512d GQA, no VE, no MTP  → 19.36M params, 16.82MB artifact (Blackwell)
#   11L 512d GQA, VE=3, MTP excl → 26.02M params, 18.70MB artifact (H100)
import torch

def estimate_mb(num_layers, model_dim, num_heads, num_kv_heads, mlp_mult,
                vocab_size=1024, bigram_vocab=2048, bigram_dim=128,
                ve_layers=0, ve_dim=128, tie_embeddings=True):
    head_dim = model_dim // num_heads
    kv_dim   = num_kv_heads * head_dim

    # ── Embeddings ────────────────────────────────────────────────────────────
    base_emb    = vocab_size * model_dim          # input embed (tied to output head)
    bigram_emb  = bigram_vocab * bigram_dim        # BigramHashEmbedding table
    bigram_proj = bigram_dim * model_dim           # bigram → model_dim projection

    # ── Per transformer layer ─────────────────────────────────────────────────
    # Attention: Q(d×d) + K(kv_dim×d) + V(kv_dim×d) + O(d×d)
    attn = model_dim**2 + 2*(kv_dim * model_dim) + model_dim**2
    # MLP: activation=relu² uses 2 matrices (fc1 + fc2), NOT 3 like SwiGLU
    mlp  = 2 * model_dim * (model_dim * mlp_mult)
    # Layer norms (scale only, 2 per block)
    ln   = 2 * model_dim
    per_layer = attn + mlp + ln

    # ── VE: shared ValueEmbedding (lookup + one projection, not per-layer) ────
    ve = (vocab_size * ve_dim + ve_dim * kv_dim) if ve_layers > 0 else 0

    # ── Misc (smear gate, QK gains, XSA, rope, etc.) ~2-3% overhead ──────────
    misc = num_layers * 1500 + 3000

    raw_params  = base_emb + bigram_emb + bigram_proj + num_layers * per_layer + ve + misc
    # Empirical correction: formula overestimates by ~3% vs measured
    total_params = int(raw_params * 0.97)

    int8_mb = total_params / 1e6   # 1 byte per weight after int8 quantization

    # Compression ratio from zlib on int8 weights — calibrated to measured runs:
    #   8L (19.36MB int8) → 16.82MB: ratio 0.869
    #  11L (26.02MB int8) → 18.70MB: ratio 0.719
    # Linear interpolation, clamped at both ends
    if int8_mb <= 19.36:
        ratio = 0.869
    elif int8_mb >= 26.02:
        ratio = 0.719
    else:
        t = (int8_mb - 19.36) / (26.02 - 19.36)
        ratio = 0.869 + t * (0.719 - 0.869)

    compressed_mb = int8_mb * ratio
    total_mb      = compressed_mb + 0.059   # + ~59KB code

    return dict(params=total_params, int8_mb=int8_mb,
                compressed_mb=compressed_mb, total_mb=total_mb,
                fits=total_mb <= 16.0, headroom_mb=16.0 - total_mb)

print("=" * 68)
print("MODEL SIZE ESTIMATOR  (int8+zlib, 16MB budget)")
print("=" * 68)
print(f"{'Config':<28} {'Params':>9}  {'int8':>6}  {'artifact':>9}  {'fits?':>8}  {'headroom':>9}")
print("-" * 68)

configs = [
    # (label, kwargs)
    ("6L 384d MHA  no VE",   dict(num_layers=6,  model_dim=384, num_heads=6, num_kv_heads=6, mlp_mult=3, ve_layers=0)),
    ("6L 512d MHA  no VE",   dict(num_layers=6,  model_dim=512, num_heads=8, num_kv_heads=8, mlp_mult=3, ve_layers=0)),
    ("6L 512d GQA  VE=3",    dict(num_layers=6,  model_dim=512, num_heads=8, num_kv_heads=4, mlp_mult=3, ve_layers=3)),
    ("7L 512d GQA  VE=3",    dict(num_layers=7,  model_dim=512, num_heads=8, num_kv_heads=4, mlp_mult=3, ve_layers=3)),
    ("8L 512d GQA  no VE",   dict(num_layers=8,  model_dim=512, num_heads=8, num_kv_heads=4, mlp_mult=3, ve_layers=0)),
    ("8L 512d GQA  VE=3  ← measured 16.82", dict(num_layers=8,  model_dim=512, num_heads=8, num_kv_heads=4, mlp_mult=3, ve_layers=3)),
    ("9L 512d GQA  VE=3",    dict(num_layers=9,  model_dim=512, num_heads=8, num_kv_heads=4, mlp_mult=3, ve_layers=3)),
    ("11L 512d GQA VE=3  ← measured 18.70", dict(num_layers=11, model_dim=512, num_heads=8, num_kv_heads=4, mlp_mult=3, ve_layers=3)),
]

for name, cfg in configs:
    r = estimate_mb(**cfg)
    fits = "✓" if r['fits'] else "✗ OVER"
    print(f"{name:<28} {r['params']:>9,}  {r['int8_mb']:>5.1f}MB  {r['total_mb']:>7.2f}MB  {fits:>8}  {r['headroom_mb']:>+8.2f}MB")

print()
print("MTP head (~524KB) excluded from serialization — not counted above.")
print("FA3 + records script uses int6+lzma: 11L fits at ~15.2MB (separate estimator).")


In [ ]:
# Cell 6: Train
import subprocess, sys, os, torch, re

if os.path.isdir('/content'):
    BASE = '/content'
elif os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
else:
    BASE = os.path.expanduser('~')

REPO = f'{BASE}/parameter-golf'
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
is_hopper_exact   = cc[0] == 9
is_ampere_plus    = cc[0] >= 8
is_blackwell_plus = cc[0] >= 12
print(f"GPU: {gpu_name}  compute_cap: {cc[0]}.{cc[1]}")

fa3_available = False
try:
    from flash_attn_interface import flash_attn_func
    fa3_available = True
except ImportError:
    pass

if is_hopper_exact and fa3_available:
    SCRIPT = f'{REPO}/records/track_10min_16mb/2026-03-26_ImprovedHyperparams/train_gpt.py'
    print("Using ImprovedHyperparams records script (FA3 + int6+lzma + n-gram)")
else:
    SCRIPT = f'{REPO}/train_gpt.py'
    print("Using root train_gpt.py (int8+zlib)")
print(f"Script: {SCRIPT}")

if is_blackwell_plus:
    os.environ['TORCHDYNAMO_DISABLE'] = '1'
    print("TORCHDYNAMO_DISABLE=1 (Blackwell SM120+, Triton not tuned)")

env = os.environ.copy()
env.update({
    'RUN_ID':                'colab_run',
    'MAX_WALLCLOCK_SECONDS': '600',
    'MUON_BACKEND_STEPS':    '10',
    'XSA_LAST_N':            '99',
    'BIGRAM_VOCAB_SIZE':     '2048',
    'SMEAR_ENABLED':         '1',
    'LN_SCALE':              '1',
    'ROPE_DIMS':             '16',
    'VE_ENABLED':            '1',
    'MTP_NUM_HEADS':         '1',
    'VAL_LOSS_EVERY':        '500',
})

if is_hopper_exact and fa3_available:
    # Records script manages its own model config.
    # Force n-gram on (defaults off for single-GPU).
    print("Model: 11L 512d GQA (records script default)")
    env.update({'NGRAM_CACHE':'1','NGRAM_ALPHA':'0.40','NGRAM_ORDER':'7'})

elif is_blackwell_plus:
    # int8+zlib budget: 7L 512d + VE ≈ 15.3MB estimated
    print("Model: 7L 512d GQA 8H/4KV")
    env.update({'NUM_LAYERS':'7','MODEL_DIM':'512','NUM_HEADS':'8','NUM_KV_HEADS':'4',
                'TRAIN_BATCH_TOKENS':'131072','VAL_BATCH_SIZE':'131072','MLP_MULT':'3',
                'VE_LAYERS':'5,6,7'})

elif is_hopper_exact:
    # H100 without FA3 — int8+zlib limits us to ~6L 512d to fit 16MB.
    # (11L = 18.7MB, 8L = 17.0MB, 7L = 16.4MB — all over; 6L ≈ 15.8MB fits)
    # Use smaller batch to get more steps and compensate for smaller model.
    print("Model: 6L 512d MHA (H100, no FA3 — int8+zlib budget constraint)")
    env.update({'NUM_LAYERS':'6','MODEL_DIM':'512','NUM_HEADS':'8','NUM_KV_HEADS':'8',
                'TRAIN_BATCH_TOKENS':'262144','VAL_BATCH_SIZE':'131072','MLP_MULT':'3',
                'VE_LAYERS':'4,5,6'})

elif is_ampere_plus:
    print("Model: 9L 512d GQA 8H/4KV")
    env.update({'NUM_LAYERS':'9','MODEL_DIM':'512','NUM_HEADS':'8','NUM_KV_HEADS':'4',
                'TRAIN_BATCH_TOKENS':'262144','VAL_BATCH_SIZE':'131072','MLP_MULT':'3',
                'VE_LAYERS':'7,8,9'})

else:
    print("Model: 6L 384d MHA")
    env.update({'NUM_LAYERS':'6','MODEL_DIM':'384','NUM_HEADS':'6','NUM_KV_HEADS':'6',
                'TRAIN_BATCH_TOKENS':'131072','VAL_BATCH_SIZE':'65536','MLP_MULT':'3',
                'VE_LAYERS':'4,5,6'})

LOG = f'{BASE}/train_log.txt'
cmd = ['torchrun','--standalone','--nproc_per_node=1', SCRIPT]
print(f"Log: {LOG}\n")

with open(LOG, 'w') as lf:
    proc = subprocess.Popen(cmd, env=env, cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush(); lf.write(line)
    proc.wait()
print(f"Exit code: {proc.returncode}")


In [ ]:
# Cell 7: Parse results from log
import re, os

if os.path.isdir('/content'):
    BASE = '/content'
elif os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
else:
    BASE = os.path.expanduser('~')

LOG = f'{BASE}/train_log.txt'
try:
    txt = open(LOG).read()
except FileNotFoundError:
    txt = ''
    print('Log not found — did Cell 6 complete?')

print('=' * 55)
print('RESULTS')
print('=' * 55)

# Final BPB — look for the last occurrence
bpb_all = re.findall(r'val_bpb:([\d.]+)', txt)
print(f'Final val_bpb  : {bpb_all[-1] if bpb_all else "not found"}')

# Sliding window (SOTA script only)
sw = re.findall(r'final_int6_sliding_window_s64_exact.*val_bpb:([\d.]+)', txt)
if sw:
    print(f'Sliding window : {sw[-1]} (stride=64)')

# Artifact size
art = re.findall(r'Total submission size.*?([\d,]+) bytes', txt)
if art:
    b = int(art[-1].replace(',',''))
    print(f'Artifact size  : {b:,} bytes ({b/1e6:.2f} MB / 16.00 MB budget)')

# Key milestone activation
print()
for label, pat in [
    ('Late QAT activated',   r'late_qat:enabled step:(\d+)'),
    ('SWA started',          r'swa:start step:(\d+)'),
    ('muon_steps',           r'muon_steps:(\d+)'),
]:
    m = re.search(pat, txt)
    print(f'{label:20s}: {m.group(1) if m else "not found in log"}')

# Last 20 lines
print()
print('--- last 20 log lines ---')
for l in txt.strip().split('\n')[-20:]:
    print(l)


## Notes

### What this notebook actually does
This is a **launcher** — the ~1500-line training code lives in the `.py` files. The notebook configures hyperparameters as env vars and calls `torchrun` via subprocess, streaming output in real-time.

### Kaggle vs Colab
- Paths are auto-detected (`/content` for Colab, `/kaggle/working` for Kaggle)
- On Kaggle: enable **Settings → Internet** before running Cell 4
- Kaggle gives **30 GPU-hrs/week free** (T4 or P100)
- Colab Free gives T4 with session time limits (~1-2 hrs)

### Why T4/P100 scores are weak
T4 is ~10x slower than H100 for this workload. A 10-min T4 run processes ~600K tokens vs ~78M on 8×H100. The model barely starts to converge. Use T4/P100 to **verify the pipeline runs without errors**, not for competition scores.

### For a real leaderboard submission (8×H100 on RunPod)
1. Flash Attention 3 is **required** for the SOTA records script:
   ```bash
   pip install flash_attn_3 --find-links https://windreamer.github.io/flash-attention3-wheels/cu128_torch291
   ```
2. Use `--train-shards 80` when downloading data
3. Use `--nproc_per_node=8`
4. Our improved submission is at `records/track_10min_16mb/2026-03-26_ImprovedHyperparams/train_gpt.py`

### N-gram interpolation (validated on 8×H100)
The ImprovedHyperparams records script includes n-gram interpolation at eval time:
- Builds a multi-order backoff n-gram cache (orders 2–7) from training tokens
- Entropy-adaptive alpha: blends n-gram more heavily on predictable tokens
- **Validated**: alpha=0.40 order=7 → **1.0336 BPB** (vs 1.0517 without optimal params)
- Does **not** add to artifact size — cache is in-memory only, not serialized
- Auto-enabled on multi-GPU (8×H100); Cell 6 forces it on for single H100 too

### SOTA comparison
| Submission | BPB | Notes |
|-----------|-----|-------|
| SOTA (2026-03-24) | 1.1154 | 11L 512d, 8×H100, FA3, GPTQ int6 |
| Our pr638 base + n-gram | 1.0336 | Validated on 8×H100 with alpha=0.40 order=7 |
| Our target (ImprovedHyperparams + n-gram) | <1.03 | Must beat SOTA by 0.0072 to qualify |
| Colab Blackwell (7L, int8+zlib) | ~1.33–1.37 | Single GPU, no FA3, for pipeline testing only |
